# Sistema Inteligente de Recomendação para Redução do Desperdício Alimentar

## Prova de conceito alinhada ao ODS 12

Este notebook foi preparado como apoio para apresentação final do projeto. Ele utiliza os resultados já gerados em `results/`, evitando reprocessar toda a base durante a apresentação.

## 1. Objetivo do Projeto

O objetivo é desenvolver uma prova de conceito de um sistema de recomendação de receitas que ajude o usuário a aproveitar ingredientes disponíveis e, com isso, contribuir para a redução do desperdício alimentar.

A proposta está alinhada ao **ODS 12 - Consumo e Produção Responsáveis**.

O projeto implementa:

- análise exploratória da base Food.com;
- baselines de predição de rating;
- filtragem colaborativa com SVD;
- filtragem baseada em conteúdo com TF-IDF sobre ingredientes;
- modelo híbrido com fator de validade/urgência simulada;
- avaliação com métricas de ranking.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS_DIR = PROJECT_ROOT / "results"
GRAFICOS_DIR = RESULTS_DIR / "graficos"

def carregar_csv(nome):
    return pd.read_csv(RESULTS_DIR / nome)

def mostrar_grafico(nome):
    display(Image(filename=str(GRAFICOS_DIR / nome)))

## 2. Base Utilizada

Foi utilizada a base **Food.com Recipes and Interactions**, principalmente os arquivos:

- `RAW_recipes.csv`: dados das receitas e ingredientes;
- `RAW_interactions.csv`: interações entre usuários e receitas, incluindo ratings.

A base não possui dados reais de estoque doméstico nem validade dos alimentos do usuário. Por isso, a urgência de validade foi tratada como uma entrada simulada ou informada pelo usuário.

In [ ]:
resumo_eda = carregar_csv("resumo_eda.csv")
resumo_eda

## 3. Análise Exploratória

A EDA mostrou uma base grande e bastante esparsa. A maioria das combinações possíveis entre usuários e receitas não possui interação observada, o que é comum em sistemas de recomendação.

Também foi observada forte concentração em ratings altos, especialmente nota 5.

In [ ]:
distribuicao = carregar_csv("distribuicao_avaliacoes.csv")
distribuicao

In [ ]:
mostrar_grafico("distribuicao_avaliacoes.png")

In [ ]:
mostrar_grafico("interacoes_por_ano.png")

## 4. Baselines de Predição de Rating

Antes dos modelos de ranking, foram calculados baselines simples de predição de rating:

- média global;
- média do usuário com fallback global;
- média da receita com fallback global.

As métricas usadas foram **RMSE** e **MAE**.

In [ ]:
metricas_baseline = carregar_csv("metricas_baseline.csv")
metricas_baseline

In [ ]:
mostrar_grafico("comparacao_rmse_mae.png")

## 5. Amostra de Modelagem

Para viabilizar o treinamento e a avaliação em ambiente local, foi usada uma amostra filtrada da base.

Critérios aplicados:

- remover ratings nulos;
- manter usuários com pelo menos 5 interações;
- manter receitas com pelo menos 10 avaliações;
- limitar quantidade máxima de usuários e receitas;
- usar `random_state=42` para reprodutibilidade.

Essa limitação deve ser mencionada na apresentação.

In [ ]:
resumo_amostra = carregar_csv("resumo_amostra_modelagem.csv")
resumo_amostra

## 6. Modelos Implementados

Foram avaliados quatro modelos de recomendação top-k:

1. **Popularidade**: recomenda receitas mais populares no treino.
2. **Conteúdo TF-IDF**: usa ingredientes das receitas e perfil do usuário baseado em receitas bem avaliadas.
3. **Colaborativo SVD**: usa matriz usuário-receita esparsa e `TruncatedSVD`.
4. **Híbrido**: combina score colaborativo, score de conteúdo e score de validade simulada.

No modelo híbrido, os pesos usados foram:

- colaborativo: 0,45;
- conteúdo: 0,35;
- validade: 0,20.

## 7. Métricas de Ranking

A avaliação top-k considera relevantes no teste as receitas com `rating >= 4`.

Métricas utilizadas:

- Precision@10;
- Recall@10;
- F1@10;
- NDCG@10;
- HitRate@10.

In [ ]:
metricas_ranking = carregar_csv("metricas_ranking_modelos.csv")
metricas_ranking

In [ ]:
mostrar_grafico("comparacao_precision_recall_ndcg.png")

In [ ]:
mostrar_grafico("comparacao_modelos_ranking.png")

## 8. Exemplo de Recomendação Híbrida

O exemplo abaixo mostra recomendações híbridas para um usuário da amostra.

O score final combina:

- score colaborativo;
- score baseado em conteúdo;
- score de validade/urgência simulada.

Exemplo de urgência simulada:

```python
{
    "banana": 5,
    "milk": 4,
    "eggs": 3,
    "flour": 2,
}
```

In [ ]:
recomendacoes = carregar_csv("recomendacoes_hibridas_exemplo.csv")
colunas = [
    "user_id",
    "recipe_id",
    "recipe_name",
    "score_final",
    "score_colaborativo_norm",
    "score_conteudo_norm",
    "score_validade_norm",
    "matched_ingredients",
    "justificativa",
]
recomendacoes[colunas]

## 9. Interpretação dos Resultados

Na amostra avaliada, o modelo de popularidade apresentou os melhores valores nas métricas de ranking. Isso indica que receitas populares são fortes candidatas em uma base com grande concentração de avaliações positivas.

O modelo híbrido não superou a popularidade nessa amostra, mas é o mais alinhado conceitualmente ao problema do projeto, pois combina personalização, ingredientes e urgência de validade simulada.

Portanto, o resultado principal não é afirmar que o híbrido é o melhor em todas as métricas, mas demonstrar uma arquitetura funcional e interpretável para recomendar receitas considerando desperdício alimentar.

## 10. Limitações

- A avaliação de ranking foi feita em amostra filtrada por restrição computacional.
- A validade dos alimentos não existe na base Food.com; ela foi simulada ou informada pelo usuário.
- O TF-IDF usa texto dos ingredientes, sem interpretação semântica profunda.
- O SVD foi treinado na matriz esparsa da amostra, não na base completa.
- Não houve teste com usuários reais ou validação A/B.

## 11. Conclusão

O projeto entrega uma prova de conceito funcional de recomendação de receitas alinhada ao ODS 12.

Foram implementadas EDA, baselines de rating, filtragem colaborativa com SVD, filtragem baseada em conteúdo com TF-IDF, modelo híbrido com validade simulada e avaliação com métricas top-k.

Como trabalhos futuros, recomenda-se coletar dados reais de despensa e validade, melhorar a normalização dos ingredientes, testar amostras maiores, ajustar os pesos do modelo híbrido e avaliar a solução com usuários reais.